In [4]:
import tensorflow as tf
print("TF version:", tf.__version__)
print("Physical devices:", tf.config.list_physical_devices())


TF version: 2.10.0
Physical devices: [PhysicalDevice(name='/physical_device:CPU:0', device_type='CPU'), PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU'), PhysicalDevice(name='/physical_device:GPU:1', device_type='GPU')]


In [3]:
import os
import numpy as np
from tqdm import tqdm

npz_folder = 'Insecta/NPZ/'

X_total = []
y_total = []
# Recorremos todos los archivos .npz en el directorio
for file in tqdm(os.listdir(npz_folder)):
    if file.endswith('.npz'):
        path = os.path.join(npz_folder, file)
        try:
            data = np.load(path)
            spec = data['spec']
            label = data['label'].item() 

            # Solo guardamos espectrogramas con forma exacta (1025, 313)
            if spec.shape == (1025, 313):
                X_total.append(spec)
                y_total.append(label)
                np.savez_compressed('train.npz', spec=spec, label=label)
            else:
                print(f"Forma inválida en: {file} -> {spec.shape}")
        except Exception as e:
            print(f"Error leyendo {file}: {e}")

# Se cargaron
print(f"\nEspectrogramas válidos cargados: {len(X_total)}")


 17%|█▋        | 600/3604 [00:23<01:38, 30.36it/s]

Forma inválida en: Daedadelus_waehnerorum__CSA34196_chunk0.npz -> (1025, 148)


 20%|██        | 724/3604 [00:27<01:32, 31.01it/s]

Forma inválida en: Copiphora_gracilis_CSA34185_chunk0.npz -> (1025, 281)


 23%|██▎       | 829/3604 [00:31<01:24, 32.71it/s]

...

Forma inválida en: Copiphora_gracilis_CSA34189_chunk0.npz -> (1025, 228)


 72%|███████▏  | 2608/3604 [01:38<00:31, 31.58it/s]

Forma inválida en: Daedadelus_waehnerorum__CSA34198_chunk0.npz -> (1025, 125)


100%|██████████| 3604/3604 [02:15<00:00, 26.64it/s]


Espectrogramas válidos cargados: 3596


In [4]:
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from keras.utils import to_categorical
from collections import Counter


# Convertir a arrays
X = np.array(X_total)
y = np.array(y_total)

# Filtrar clases con menos de 2 muestras
label_counts = Counter(y)
clases_validas = [label for label, count in label_counts.items() if count >= 2]

X = np.array([x for x, label in zip(X, y) if label in clases_validas])
y = np.array([label for label in y if label in clases_validas])

print(f"Clases restantes tras filtrar: {set(y)}")

# Codificar etiquetas
le = LabelEncoder()
y_encoded = le.fit_transform(y)
y_cat = to_categorical(y_encoded)

# Normalizar y reshaping para CNN
X = X / np.max(X)  # normalización
X = X[..., np.newaxis]  # añadir canal

# Separar entrenamiento y validación
X_train, X_val, y_train, y_val = train_test_split(
    X, y_cat, test_size=0.2, stratify=y_encoded, random_state=42
)


Clases restantes tras filtrar: {'Cicadidae', 'Neoconocephalus_brachypterus', 'Orophus_conspersus', 'Copiphora_colombiae', 'Tettigoniidae', 'Ragoniella_pulchella', 'Typophyllum_inflatum', 'Panoploscelis_specularis', 'Gryllidae', 'Eschatoceras_bipunctatus', 'Copris_susanae', 'Docidocercus_fasciatus', 'Oxyprora_surinamensis', 'Copiphora_gracilis', 'Cocconotus_aratifrons', 'Subria_sylvestris'}


In [12]:
from keras.models import Sequential
from keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout

model = Sequential([
    Conv2D(32, (3,3), activation='relu', input_shape=(1025, 313, 1)),
    MaxPooling2D((2,2)),
    Dropout(0.25),

    Conv2D(64, (3,3), activation='relu'),
    MaxPooling2D((2,2)),
    Dropout(0.25),

    Flatten(),
    Dense(128, activation='relu'),
    Dropout(0.5),
    Dense(y_cat.shape[1], activation='softmax')  # salida según número de clases
])

model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
model.summary()


Model: "sequential_3"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 conv2d_6 (Conv2D)           (None, 1023, 311, 32)     320       
                                                                 
 max_pooling2d_6 (MaxPooling  (None, 511, 155, 32)     0         
 2D)                                                             
                                                                 
 dropout_9 (Dropout)         (None, 511, 155, 32)      0         
                                                                 
 conv2d_7 (Conv2D)           (None, 509, 153, 64)      18496     
                                                                 
 max_pooling2d_7 (MaxPooling  (None, 254, 76, 64)      0         
 2D)                                                             
                                                                 
 dropout_10 (Dropout)        (None, 254, 76, 64)      

In [ ]:
history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=30,
    batch_size=16
)


Epoch 1/30


2025-07-24 18:12:17.544564: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:114] Plugin optimizer for device_type GPU is enabled.
2025-07-24 18:12:17.647791: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:306] Could not identify NUMA node of platform GPU ID 0, defaulting to 0. Your kernel may not have been built with NUMA support.
2025-07-24 18:12:17.647866: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:306] Could not identify NUMA node of platform GPU ID 1, defaulting to 0. Your kernel may not have been built with NUMA support.
2025-07-24 18:12:17.647960: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:272] Created TensorFlow device (/job:localhost/replica:0/task:0/device:GPU:0 with 40595 MB memory) -> physical PluggableDevice (device: 0, name: DML, pci bus id: <undefined>)
2025-07-24 18:12:17.648102: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_f

180/180 [==============================] - ETA: 0s - loss: 1.8877 - accuracy: 0.4694

2025-07-24 18:16:49.968486: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:114] Plugin optimizer for device_type GPU is enabled.
2025-07-24 18:16:50.018344: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:306] Could not identify NUMA node of platform GPU ID 0, defaulting to 0. Your kernel may not have been built with NUMA support.
2025-07-24 18:16:50.018401: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:306] Could not identify NUMA node of platform GPU ID 1, defaulting to 0. Your kernel may not have been built with NUMA support.
2025-07-24 18:16:50.018444: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:272] Created TensorFlow device (/job:localhost/replica:0/task:0/device:GPU:0 with 40595 MB memory) -> physical PluggableDevice (device: 0, name: DML, pci bus id: <undefined>)
2025-07-24 18:16:50.018462: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_f

180/180 [==============================] - 286s 2s/step - loss: 1.8877 - accuracy: 0.4694 - val_loss: 1.1624 - val_accuracy: 0.6106
Epoch 2/30
180/180 [==============================] - 279s 2s/step - loss: 1.1057 - accuracy: 0.6384 - val_loss: 0.9531 - val_accuracy: 0.7038
Epoch 3/30
180/180 [==============================] - 282s 2s/step - loss: 0.8590 - accuracy: 0.7149 - val_loss: 0.8130 - val_accuracy: 0.7163
Epoch 4/30
180/180 [==============================] - 282s 2s/step - loss: 0.6998 - accuracy: 0.7542 - val_loss: 0.6950 - val_accuracy: 0.7636
Epoch 5/30
180/180 [==============================] - 281s 2s/step - loss: 0.5437 - accuracy: 0.8115 - val_loss: 0.6419 - val_accuracy: 0.7942
Epoch 6/30
180/180 [==============================] - 281s 2s/step - loss: 0.4942 - accuracy: 0.8230 - val_loss: 0.6170 - val_accuracy: 0.8039
Epoch 7/30
180/180 [==============================] - 283s 2s/step - loss: 0.4171 - accuracy: 0.8515 - val_loss: 0.6510 - val_accuracy: 0.7942
Epoch 8/30

In [ ]:
model.save("modelo_entrenado_insecta4.h5")
